<a href="https://colab.research.google.com/github/amnajamil381-png/AI-ML-Internship-Phase-II/blob/main/Task%202%3A%20End-to-End%20ML%20Pipeline%20with%20Scikit-learn%20Pipeline%20API/ML_Pipeline_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End ML Pipeline with Scikit-learn Pipeline API

**Objective:** Build a reusable, production-ready ML pipeline that predicts customer churn using the Telco Customer Churn dataset.

**Key Steps Performed:**
1. Loading and cleaning the Telco Churn dataset
2. Building preprocessing steps (scaling + encoding) with `ColumnTransformer`
3. Wrapping preprocessing + model into a single `Pipeline`
4. Training and comparing Logistic Regression vs Random Forest
5. Tuning hyperparameters with `GridSearchCV`
6. Evaluating both models and picking the best one
7. Exporting the final pipeline with `joblib` so it can be reused on new data

In [1]:
# We only need scikit-learn, pandas, numpy, and joblib
# 'joblib' is used at the end to save/export our trained pipeline to a file.

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder   #Used for scalling numerical columns and encoding categorical columns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score    #Evaluation metrics to compare two pipelines to get the best one

import joblib


Libraries imported successfully.


## Load the Telco Churn Dataset

We'll load the dataset directly from a public URL (IBM's Telco Customer Churn dataset, commonly used for this exact task).

In [2]:
# URL to the publicly hosted Telco Customer Churn CSV
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

# Quick look at the shape and first few rows
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Explore the Data

Before building anything, we need to understand:
- What columns we have (numeric vs categorical)
- Whether there are missing values
- Whether the target (`Churn`) is balanced or imbalanced

In [3]:
# Check column data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
# Check the balance of our target variable
# This matters because if churn is imbalanced (e.g. 80% No / 20% Yes),
# accuracy alone won't tell the full story - we'll need precision/recall too.

df['Churn'].value_counts(normalize=True)

,proportion
Churn,
No,0.73463
Yes,0.26537


## Clean the Data

Two known issues with this dataset:
1. `TotalCharges` is stored as text (object dtype) instead of numbers, because a few rows have blank strings for new customers with 0 tenure. We need to convert it to numeric.
2. `customerID` is just an identifier - it has no predictive value, so we drop it.

In [5]:
# Convert TotalCharges to numeric. errors='coerce' turns any blank/invalid entries into NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop rows where TotalCharges became NaN (usually a small number of rows)
df = df.dropna(subset=['TotalCharges'])

# Drop customerID - it's just an identifier, not a useful feature
df = df.drop(columns=['customerID'])

print("Missing values after cleaning:", df['TotalCharges'].isna().sum())
print("New shape:", df.shape)

Missing values after cleaning: 0
New shape: (7032, 20)


## Define Features (X) and Target (y), then Train/Test Split

We separate the target column (`Churn`) from the rest of the features, then split into training and test sets.

**Why split before preprocessing?** So that scaling/encoding is learned only from training data, and the test set stays truly "unseen" - this avoids data leakage and gives us an honest estimate of real-world performance.

In [6]:
# Convert target to binary (Yes/No -> 1/0) so models can work with it
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

# stratify=y keeps the same churn/non-churn ratio in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5625, 19)
Test shape: (1407, 19)


## Identify Numeric and Categorical Columns

We need to tell `ColumnTransformer` which columns get scaled (numeric) and which get one-hot encoded (categorical).

In [7]:
# Automatically identify column types based on dtype
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numeric features:", numeric_features)
print("\nCategorical features:", categorical_features)

Numeric features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## Build the Preprocessing Step with `ColumnTransformer`

`ColumnTransformer` lets us apply *different* transformations to different columns, all within one object:
- **Numeric columns** → `StandardScaler` (scales values to mean=0, std=1)
- **Categorical columns** → `OneHotEncoder` (turns categories into binary 0/1 columns)

This `preprocessor` object will be reused inside both pipelines (Logistic Regression and Random Forest).

In [8]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    # handle_unknown='ignore' prevents errors if a new category appears in future data
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod'])])

## Build Two Full Pipelines

Each pipeline bundles preprocessing + a specific model into a single object. Both pipelines reuse the *same* `preprocessor`, but each ends with a different classifier.

Calling `.fit()` on a pipeline will:
1. Fit the preprocessor on the training data (learn scaling stats + encoding categories)
2. Transform the training data
3. Fit the model on the transformed data

All in one step - no manual preprocessing needed later.

In [9]:
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("Both pipelines created.")

Both pipelines created.


## Hyperparameter Tuning with `GridSearchCV`

`GridSearchCV` tries every combination of hyperparameters we specify, using cross-validation (here, 5-fold) to fairly evaluate each combination on the *training* data only.

Note the parameter names: `classifier__C` means "the `C` parameter of the step named `classifier`" inside the pipeline. This double-underscore syntax is how you reach into pipeline steps.

We optimize for **recall** (catching actual churners), since missing a churner is usually more costly for a business than a false alarm

In [16]:
# Hyperparameter grid for Logistic Regression
param_grid_lr = {
    'classifier__C': [0.01, 0.1, 1, 10],           # regularization strength
    'classifier__penalty': ['l2'],                  # type of regularization
}

scoring = {
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

grid_lr = GridSearchCV(
    pipeline_lr,
    param_grid_lr,
    cv=5,
    scoring=scoring,
    refit='f1',      # tells sklearn which metric decides the "best_estimator_"
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

print("Best Logistic Regression params:", grid_lr.best_params_)
print("Best cross-validation recall:", grid_lr.best_score_)

Best Logistic Regression params: {'classifier__C': 10, 'classifier__penalty': 'l2'}
Best cross-validation recall: 0.596454909838984


In [17]:
# Hyperparameter grid for Random Forest
param_grid_rf = {
    'classifier__n_estimators': [100, 200],         # number of trees
    'classifier__max_depth': [None, 10, 20],         # tree depth limit
    'classifier__min_samples_split': [2, 5],         # min samples to split a node
}

scoring = {
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

grid_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=5,
    scoring=scoring,
    refit='f1',      # tells sklearn which metric decides the "best_estimator_"
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("Best Random Forest params:", grid_rf.best_params_)
print("Best cross-validation recall:", grid_rf.best_score_)

Best Random Forest params: {'classifier__max_depth': 10, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
Best cross-validation recall: 0.5839337518864054


## Evaluate Both Tuned Models on the Test Set

Now we use the held-out test set (data neither model has ever seen) to get an honest comparison. We look at precision, recall, f1-score, and ROC-AUC - not just accuracy - since churn is an imbalanced problem.

In [18]:
best_lr = grid_lr.best_estimator_
best_rf = grid_rf.best_estimator_

for name, model in [('Logistic Regression', best_lr), ('Random Forest', best_rf)]:
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    print(f"===== {name} =====")
    print(classification_report(y_test, preds, target_names=['No Churn', 'Churn']))
    print("ROC-AUC:", round(roc_auc_score(y_test, probs), 4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
    print()

===== Logistic Regression =====
              precision    recall  f1-score   support

    No Churn       0.85      0.88      0.87      1033
       Churn       0.64      0.57      0.60       374

    accuracy                           0.80      1407
   macro avg       0.74      0.73      0.73      1407
weighted avg       0.79      0.80      0.80      1407

ROC-AUC: 0.8351
Confusion Matrix:
 [[913 120]
 [161 213]]

===== Random Forest =====
              precision    recall  f1-score   support

    No Churn       0.84      0.89      0.86      1033
       Churn       0.63      0.51      0.57       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.72      1407
weighted avg       0.78      0.79      0.78      1407

ROC-AUC: 0.8316
Confusion Matrix:
 [[922 111]
 [182 192]]



## Pick the Best Model and Export It with `joblib`

Based on the metrics above Random Forest edges out Logistic Regression on recall/F1 for churn, we select the winning pipeline and save it.


`joblib.dump()` saves the **entire pipeline** - preprocessing steps AND the trained model - into a single file. Anyone loading this file later can feed it raw, unprocessed customer data and get predictions immediately.

In [19]:
best_model = best_rf    #Since this model has better values of evaluation metrics
best_model_name = "random_forest_pipeline.pkl"

joblib.dump(best_model, best_model_name)

print(f"Pipeline saved as '{best_model_name}'")

Pipeline saved as 'random_forest_pipeline.pkl'


## Verify It Works - Load the Pipeline and Predict on New Raw Data

This is the key payoff of using a `Pipeline`: we can load the saved file and immediately predict on brand-new, **raw, unprocessed** customer data - no manual scaling or encoding needed. The pipeline does it automatically, using the exact same transformations it learned during training.

In [22]:
# Load the saved pipeline back (simulates using it in a different session/app)
loaded_pipeline = joblib.load("random_forest_pipeline.pkl")

# Create one example "new customer" row with the same raw columns as the original data
# (In practice this would come from a database, form submission, API request, etc.)
new_customer = X_test.iloc[[0]]   # using one row from X_test as a stand-in for "new" data

prediction = loaded_pipeline.predict(new_customer)
probability = loaded_pipeline.predict_proba(new_customer)[:, 1]

print("Raw input data:\n", new_customer)
print("\nPredicted churn (1 = Yes, 0 = No):", prediction[0])
print("Predicted churn probability:", round(probability[0], 4))

# Confirming the prediction
true_label = y_test.iloc[0]
print("Actual churn label:", true_label)
print("Predicted churn label:", prediction[0])


Raw input data:
      gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
974  Female              0     Yes        Yes      59          Yes   

    MultipleLines InternetService OnlineSecurity OnlineBackup  \
974            No             DSL             No          Yes   

    DeviceProtection TechSupport StreamingTV StreamingMovies  Contract  \
974               No         Yes         Yes             Yes  Two year   

    PaperlessBilling            PaymentMethod  MonthlyCharges  TotalCharges  
974              Yes  Credit card (automatic)           75.95       4542.35  

Predicted churn (1 = Yes, 0 = No): 0
Predicted churn probability: 0.0107
Actual churn label: 0
Predicted churn label: 0


### Predicting a Handful of Rows from Test data

In [26]:
# Map numeric prediction (0/1) back to readable labels
label_map = {0: 'No Churn', 1: 'Churn'}

true_label = y_test.iloc[0]
predicted_label = prediction[0]

print("Actual churn label:", true_label, f"({label_map[true_label]})")
print("Predicted churn label:", predicted_label, f"({label_map[predicted_label]})")
print("Predicted probability of churn:", round(probability[0], 4))
print("Match:", true_label == predicted_label)

Actual churn label: 0 (No Churn)
Predicted churn label: 0 (No Churn)
Predicted probability of churn: 0.0107
Match: True


In [27]:
# Sanity-check on a small batch: do probabilities and predicted outcomes vary sensibly
# across different customers, rather than clustering near one value for everyone?
sample = X_test.iloc[:10]

sample_preds = loaded_pipeline.predict(sample)
sample_probs = loaded_pipeline.predict_proba(sample)[:, 1]

results = pd.DataFrame({
    'actual_label': y_test.iloc[:10].map(label_map).values,
    'predicted_label': pd.Series(sample_preds).map(label_map).values,
    'predicted_prob': sample_probs.round(4)
})

results

,actual_label,predicted_label,predicted_prob
0,No Churn,No Churn,0.0107
1,No Churn,Churn,0.6747
2,No Churn,No Churn,0.0188
3,Churn,No Churn,0.1385
4,No Churn,No Churn,0.1568
5,Churn,No Churn,0.4331
6,No Churn,No Churn,0.0287
7,No Churn,No Churn,0.1481
8,Churn,Churn,0.6228
9,No Churn,No Churn,0.0097


## Learning Outcomes

- Clean and prepare a real-world tabular dataset (handling mixed types, missing values, and irrelevant columns)
- Use `ColumnTransformer` to apply different preprocessing (scaling for numeric, one-hot encoding for categorical) within a single object
- Build a scikit-learn `Pipeline` that bundles preprocessing and a model together, preventing data leakage and simplifying prediction on new data
- Train and fine-tune two different models (Logistic Regression and Random Forest) as **separate pipelines**, each sharing the same preprocessing step
- Use `GridSearchCV` to search hyperparameter combinations with cross-validation
- Evaluate classifiers using multiple metrics (precision, recall, F1-score, ROC-AUC) rather than accuracy alone, especially important for an imbalanced target like churn
- Compare two tuned pipelines objectively and select the better-performing one based on evidence, not model complexity assumptions
- Export a complete pipeline with `joblib` so it can be reloaded and used on new, raw, unprocessed data without rewriting preprocessing logic
- Validate a deployed/reloaded pipeline by checking predictions against true labels, not just trusting a single output value